In [ ]:
import polars as pl
import altair as alt
import pandas #for Datawrapper
import pyarrow
import geopandas as gpd
from shapely.geometry import Point
alt.data_transformers.disable_max_rows()

Graph Ideas:
- Heat Map
- Strip Plot
- Stacked Area
Geospatial:
- Heat map of population by area
- Choropleth of county frequency

In [ ]:
prisoner_df = pl.read_csv("data/prisoner_dataset.csv")
facility_df = pl.read_csv("data/latest_facility_counts.csv")

In [ ]:
#Cleaning/merging

facility_df = facility_df.filter(
    pl.col("State") == "Texas"
).with_columns(
    pl.col("Name").str.split(by=" ").list.first(),
)

facility_df = facility_df.unique(subset=["Name"])

facility_df = facility_df.with_columns(
    pl.col("Name").str.to_titlecase().alias("Current Facility")
)

cutoff_date = pl.lit("09/22/2025").str.to_date("%m/%d/%Y") #From date publish

prisoner_df = prisoner_df.with_columns(
    pl.when(pl.col("Sentence (Years)").is_in(["Capital Life", "Life", "LWOP"]))
    .then(pl.lit("Yes"))
    .otherwise(pl.lit("No"))
    .alias("Life Sentence?")
)

prisoner_df = prisoner_df.with_columns(
    pl.col("TDCJ Offense").str.to_titlecase(),
    pl.col("Sentence Date").str.to_date("%m/%d/%Y"),
)

prisoner_df = prisoner_df.with_columns(
    (cutoff_date - pl.col("Sentence Date")).dt.total_days().alias("Time Served"),
)

prisoner_df = prisoner_df.with_columns(
    (pl.col("Time Served")/365)
)

breaks = [16, 20, 25, 30, 40, 50, 60, 70, 80]
labels = [
    "<16", "16-20", "20-25", "25-30", "30-40",
    "40-50", "50-60", "60-70", "70-80", "80+"
]

binned_female_prisoners = prisoner_df.with_columns(
    pl.col("Age").cut(breaks, labels=labels).alias("Age_bucket")
)

binned_female_prisoners = binned_female_prisoners.filter(
    (pl.col("Gender") == "F")
)

combined_df = prisoner_df.join(facility_df, on="Current Facility")

race_comparison_df = prisoner_df.filter(
    pl.col("Race").is_in(["W", "H", "B"]),
)

In [ ]:
#1. 10 Most Common Offenses in Texas Jails

def common_offenses(df):
    chart = alt.Chart(df, title="Top 10 Most Common Charges")
    domain = ["No", "Yes"]
    range = ["#b0a1a1", "#44414f"]
    common_bar_chart = chart.mark_bar().encode(
        alt.Y("TDCJ Offense:N").sort("-x"),
        alt.X("count:Q"),
        alt.Color("Life Sentence?:N").scale(domain=domain, range=range)
    ).transform_aggregate(
        count="count()",
        groupby=["TDCJ Offense", "Life Sentence?"]
    ).transform_window(
        rank="rank(count)",
        sort=[alt.SortField("count", order="descending")]
    ).transform_filter(
        (alt.datum.rank <= 10)
    )
    return common_bar_chart

common_offenses(prisoner_df)

#data could be cleaned more - see the repeat rows with Aggravated Sexual Assault of a Child

In [ ]:
#2. Age and average length of sentence

def age_sentence(df):
    chart = alt.Chart(df, title="Average Sentence Length and Time Served by Age")
    domain = ["Sentence (Years)", "Time Served"]
    range = ["#b0a1a1", "#44414f"]

    age_sentence_chart = chart.transform_filter(
        (alt.datum["Sentence (Years)"] < 100),
        (alt.datum["Age"] < 100)
    ).transform_fold(
        ["Sentence (Years)", "Time Served"],
        as_=["key", "value"]
    ).transform_aggregate(
            mean_value="mean(value)",
            groupby=["Age", "key"]
    ).mark_line().encode(
        alt.X("Age:Q"),
        alt.Y("mean_value:Q", title="Years"),
        alt.Color("key:N").scale(domain=domain, range=range)
    )

    return age_sentence_chart

#maybe rename Sentence column beforehand?

age_sentence(prisoner_df)

In [ ]:
#3. Scatterchart of prisons by number of prisoners and staff

def prison_pop_scatter(df):
    chart = alt.Chart(df, title="Prisons by Inmate and Staff Count")
    prison_pop_scatter = chart.mark_point().transform_aggregate(
        count="count()",
        staff_confirmed="average(Staff.Confirmed)",
        groupby=["Current Facility"]
    ).encode(
        alt.X("count:Q", title="Prisoner Count"),
        alt.Y("staff_confirmed:Q", title="Staff Confirmed"),
        color = alt.value("#44414f")
    )

    prison_pop_regression = prison_pop_scatter.transform_regression("count", "staff_confirmed").mark_line().encode(
        x=alt.X("count:Q"),
        y=alt.Y("staff_confirmed:Q"),
        color=alt.value("#b0a1a1")
        )
    
    combined = prison_pop_scatter + prison_pop_regression

    return combined

prison_pop_scatter(combined_df)

In [ ]:
#4. Heatmap of prisoner deaths relative to size

def prison_pop_heat(df):
    chart = alt.Chart(df, title="Inmates Deaths Relative to Inmate and Staff Counts")
    prison_pop_heat = chart.mark_rect().transform_aggregate(
        count="count()",
        staff_confirmed="average(Staff.Confirmed)",
        residents_deaths="average(Residents.Deaths)",
        groupby=["Current Facility"]
    ).encode(
        alt.X("count:Q", title="Prisoner Count"),
        alt.Y("staff_confirmed:Q", title="Staff Confirmed"),
        alt.Color("residents_deaths:Q", title="Inmate Deaths", scale=alt.Scale(scheme="viridis"))
    )

    return prison_pop_heat

prison_pop_heat(combined_df)

In [ ]:
#5. race on sentence time - stacked area

def race_sentence(df):
    chart = alt.Chart(df, title="Comparison of Sentence Length Among Black, White, and Hispanic Inmates")
    domain = ["B", "H", "W"]
    range = ["#44414f","#686576", "#b0a1a1"]
    race_sentence_chart = chart.mark_area().encode(
        alt.X("Age:Q"),
        alt.Y("average(Sentence (Years)):Q"),
        alt.Color("Race:N").scale(domain=domain, range=range)
    ).transform_filter(
        (alt.datum["Sentence (Years)"] < 100),
        (alt.datum["Age"] < 100)
    )

    return race_sentence_chart

#find way to change to full races?

race_sentence(race_comparison_df)

In [ ]:
#6. race on sentence time - violin plot

def race_sentence_violin(df):
    df = df.filter(
        pl.col('Sentence (Years)').cast(pl.Float64, strict=False).is_not_null()
    )
    domain = ["B", "H", "W"]
    range = ["#44414f","#686576", "#b0a1a1"]
    chart = alt.Chart(df, width=100, title="Comparison of Sentence Length Among Black, White, and Hispanic Inmates").transform_density(
        "Sentence (Years)",
        as_=["Sentence (Years)", "density"],
        extent = [0,100],
        groupby = ["Race"]
    ).mark_area(orient="horizontal").encode(
        alt.X("density:Q")
            .stack("center")
            .impute(None)
            .title(None)
            .axis(labels=False, grid=False, ticks=True),
        alt.Y("Sentence (Years):Q"),
        alt.Color("Race:N").scale(domain=domain, range=range),
        alt.Column("Race:N")
            .spacing(0)
            .header(titleOrient='bottom', labelOrient='bottom', labelPadding=0)
    ).configure_view(
        stroke=None
    )

    return chart

race_sentence_violin(race_comparison_df)

In [ ]:
#7. Parole review process status versus age histogram
def parole_review(df):
    domain = ["IN PAROLE REVIEW PROCESS", "NOT IN REVIEW PROCESS"]
    range = ["#44414f", "#b0a1a1"]
    chart = alt.Chart(df, title="Parole Review Status by Age Groups")
    parole_histogram = chart.mark_bar().encode(
    alt.X("Age:Q", bin=True),
    alt.Y("count():Q"),
    alt.Color("Parole Review Status:N").scale(domain=domain, range=range)
    )   
    
    return parole_histogram

#need to clean "null" data for final graphic

parole_review(prisoner_df)

In [ ]:
#8. radial plot for sentence time for women
binned_female_prisoners = binned_female_prisoners.with_columns(
    pl.col("Age_bucket").cast(pl.String)
)

def women_sentence_times(df):
    df = df.filter(
        pl.col('Sentence (Years)').cast(pl.Float64, strict=False).is_not_null()
    )
    chart = alt.Chart(df, title="Comparison of Sentence Lengths Among Female Inmates by Age Groups")
    base = chart.transform_aggregate(
            avg_sentence="average(Sentence (Years))",
            groupby=["Age_bucket"]
            ).encode(
            alt.Theta("avg_sentence:Q").stack(True),
            alt.Radius("avg_sentence:Q").scale(type="sqrt", zero=True, rangeMin=20),
            alt.Color("Age_bucket:N", legend=alt.Legend(title="Age Group"), scale=alt.Scale(scheme="viridis")),
            )

    c1 = base.mark_arc(innerRadius=20, stroke="#fff")

    c2 = base.mark_text(radiusOffset=10).encode(
        text=alt.Text("avg_sentence:Q", format='.1f'),
        color=alt.value("black"))

    final_radial = c1 + c2
    
    return final_radial

women_sentence_times(binned_female_prisoners)   

In [ ]:
#geospatial of Texas - heatmap of population by area

#AI: I was stuck on where to get started with simple geospatial mapping, so I asked ChatGPT to help me get started. See details in citations.md. 

# use https://altair-viz.github.io/gallery/choropleth.html for counties

def choropleth_facility(combined_df):

    geo_combined_df = combined_df.to_pandas()

    gdf = gpd.GeoDataFrame(
        geo_combined_df,
        geometry=gpd.points_from_xy(geo_combined_df.Latitude, geo_combined_df.Longitude),
        crs="EPSG:4326"  # WGS84 lat/lon
    )

    texas = gpd.read_file("data/State_Boundary.shp")

    ax = texas.plot(color='white', edgecolor='black', figsize=(8,8))
    gdf.plot(ax=ax, markersize=gdf['count()'], color='red', alpha=0.6)

In [ ]:
#counties of prisoner origin choropleth